# LSTM Gesture Classifier

This notebook trains a **stacked LSTM** model for dynamic gesture recognition using time-series data from a sensory data glove.

**Data format:**  
Each gesture trial is a CSV file (~5 s, ~22 Hz) stored in a folder whose name is the gesture label.  
Column naming convention: `{hand}_{segment}_{loc}_{channel}`  
e.g. `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_flex_mcp`

**Pipeline overview:**
1. Select sensor channels via config flags
2. Load + filter + resample + normalise all trials
3. Stratified train / val / test split
4. Build and train a stacked LSTM with early stopping
5. Evaluate with confusion matrix and per-class classification report
6. Save model weights and a JSON results summary

---
**Column naming convention:**
```
{hand}_{segment}_{loc}_{channel}

Examples:
  left_thumb_mid_ax          right_index_prox_pitch
  left_thumb_flex_mcp        right_palm_az
  left_wrist_roll
```
Quaternion columns (`qw`, `qx`, `qy`, `qz`) are **always excluded**.

---
## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'scipy',
    'matplotlib',
    'seaborn',
]

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs
)
print('All dependencies installed.')

---
## Cell 2 — Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
from utils.data_loader import (
    build_column_groups, HANDS, FINGERS, LOCS,
    IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS,
)

# Resolve selection flags into column lists
_hands   = ([h for h in HANDS   if (USE_LEFT_HAND   and h=="left")
                                 or (USE_RIGHT_HAND  and h=="right")]
            or HANDS)
_fingers = FINGERS
_locs    = [l for l in LOCS
            if (USE_DISTAL_IMU   and l=="mid")
            or (USE_PROXIMAL_IMU and l=="prox")] or LOCS

feature_cols = build_column_groups(
    hands          = _hands,
    fingers        = _fingers,
    locs           = _locs,
    include_flex        = USE_FLEX_SENSORS,
    include_palm_prox   = USE_PALM_IMU,
    include_wrist       = USE_WRIST_IMU,
    include_accel       = USE_ACCELEROMETER,
    include_orient      = USE_ORIENTATION,
)

print(f"Feature columns selected: {len(feature_cols)}")
print()
# Group display
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)
for grp, cols in list(groups.items())[:12]:
    print(f"  {grp:40s}  ({len(cols)} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")


---
## Cell 3 — CONFIG

> **Edit all experiment settings here.** Every tunable parameter lives in this cell — no other cell needs to be modified for standard experiments.

In [ ]:
# =============================================================================
# CONFIGURATION  —  Edit this cell to control every aspect of the pipeline
# =============================================================================

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s = 31.2 Hz)
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz); set lower
                        # (e.g. 78) to halve resolution and speed up training.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations are valid — the feature list is built automatically.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (tip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment)

# IMU channel types (applies to finger IMUs AND palm_prox AND wrist)
USE_ACCELEROMETER = True  # ax, ay, az
USE_ORIENTATION   = True  # yaw, pitch, roll  (wrist uses 'heading' instead of 'yaw')

# Flex sensors  (MCP and PIP joints per finger)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU  (palm_prox — the real palm IMU; palm_mid is always zero → auto-excluded)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied to each channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove firmware;
# set 'none' to use them as-is, or apply an additional filter here.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalization applied after resampling
# Options: 'minmax' (→ [0,1]) | 'zscore' (zero-mean unit-var) | 'none'
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalize each trial independently (recommended)

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples (auto)
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
LSTM_UNITS        = [128, 64]  # Units per stacked LSTM layer
DROPOUT_RATE      = 0.3        # Dropout after each LSTM layer
RECURRENT_DROPOUT = 0.2        # Recurrent dropout within LSTM cells
DENSE_UNITS       = [64]       # Hidden FC units before softmax output

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 15


---
## Cell 4 — Build Feature Column List from Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
from utils.data_loader import (
    build_column_groups, HANDS, FINGERS, LOCS,
    IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS,
)

# Resolve selection flags into column lists
_hands   = ([h for h in HANDS   if (USE_LEFT_HAND   and h=="left")
                                 or (USE_RIGHT_HAND  and h=="right")]
            or HANDS)
_fingers = FINGERS
_locs    = [l for l in LOCS
            if (USE_DISTAL_IMU   and l=="mid")
            or (USE_PROXIMAL_IMU and l=="prox")] or LOCS

feature_cols = build_column_groups(
    hands          = _hands,
    fingers        = _fingers,
    locs           = _locs,
    include_flex        = USE_FLEX_SENSORS,
    include_palm_prox   = USE_PALM_IMU,
    include_wrist       = USE_WRIST_IMU,
    include_accel       = USE_ACCELEROMETER,
    include_orient      = USE_ORIENTATION,
)

print(f"Feature columns selected: {len(feature_cols)}")
print()
# Group display
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)
for grp, cols in list(groups.items())[:12]:
    print(f"  {grp:40s}  ({len(cols)} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")


---
## Cell 5 — Load and Preprocess Dataset

In [ ]:
# =============================================================================
# CONFIGURATION  —  Edit this cell to control every aspect of the pipeline
# =============================================================================

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s = 31.2 Hz)
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz); set lower
                        # (e.g. 78) to halve resolution and speed up training.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations are valid — the feature list is built automatically.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (tip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment)

# IMU channel types (applies to finger IMUs AND palm_prox AND wrist)
USE_ACCELEROMETER = True  # ax, ay, az
USE_ORIENTATION   = True  # yaw, pitch, roll  (wrist uses 'heading' instead of 'yaw')

# Flex sensors  (MCP and PIP joints per finger)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU  (palm_prox — the real palm IMU; palm_mid is always zero → auto-excluded)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied to each channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove firmware;
# set 'none' to use them as-is, or apply an additional filter here.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalization applied after resampling
# Options: 'minmax' (→ [0,1]) | 'zscore' (zero-mean unit-var) | 'none'
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalize each trial independently (recommended)

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples (auto)
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
LSTM_UNITS        = [128, 64]  # Units per stacked LSTM layer
DROPOUT_RATE      = 0.3        # Dropout after each LSTM layer
RECURRENT_DROPOUT = 0.2        # Recurrent dropout within LSTM cells
DENSE_UNITS       = [64]       # Hidden FC units before softmax output

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 15


---
## Cell 6 — Train / Val / Test Split

In [ ]:
"""
Stratified split into train / val / test subsets.
Labels are one-hot encoded for Keras categorical crossentropy.
"""

os.makedirs('results', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

(X_train, y_train_int), (X_val, y_val_int), (X_test, y_test_int) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

# One-hot encode integer labels → shape (N, NUM_CLASSES)
y_train = to_categorical(y_train_int, num_classes=NUM_CLASSES)
y_val   = to_categorical(y_val_int,   num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test_int,  num_classes=NUM_CLASSES)

print('Split shapes:')
print(f'  X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}     y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}    y_test  : {y_test.shape}')

# Verify class balance in each split
for split_name, split_y in [('Train', y_train_int), ('Val', y_val_int), ('Test', y_test_int)]:
    u, c = np.unique(split_y, return_counts=True)
    balance = ', '.join(f'{CLASS_NAMES[i]}:{n}' for i, n in zip(u, c))
    print(f'  {split_name:6s} class counts: {balance}')

---
## Cell 7 — Model Definition

In [ ]:
"""
Build a stacked LSTM model.

Architecture
------------
Input  →  [LSTM(units, return_sequences=True)  →  Dropout]  × (n-1 layers)
       →   LSTM(units, return_sequences=False) →  Dropout
       →  [Dense(units, activation='relu')     →  Dropout]  × n_dense
       →   Dense(NUM_CLASSES, activation='softmax')

Notes
-----
- return_sequences=True is required for all LSTM layers except the last
  so that the next LSTM layer receives a full time-step sequence.
- recurrent_dropout > 0 disables CuDNN kernel; use 0.0 for GPU speed.
"""

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def build_lstm_model(
    input_shape,
    num_classes,
    lstm_units,
    dropout_rate,
    recurrent_dropout,
    dense_units,
):
    """
    Construct a configurable stacked LSTM classifier.

    Parameters
    ----------
    input_shape       : (timesteps, features)
    num_classes       : number of gesture classes (output units)
    lstm_units        : list[int] — units per LSTM layer
    dropout_rate      : float in [0, 1) — output dropout after each LSTM
    recurrent_dropout : float in [0, 1) — recurrent dropout inside LSTM
    dense_units       : list[int] — units per hidden FC layer

    Returns
    -------
    keras.Model
    """
    inputs = keras.Input(shape=input_shape, name='glove_input')
    x = inputs

    # ── Stacked LSTM layers ────────────────────────────────────────────────────
    for i, units in enumerate(lstm_units):
        return_seq = (i < len(lstm_units) - 1)  # True for all but the last layer
        x = layers.LSTM(
            units,
            return_sequences  = return_seq,
            dropout           = dropout_rate,
            recurrent_dropout = recurrent_dropout,
            name              = f'lstm_{i+1}',
        )(x)
        x = layers.Dropout(dropout_rate, name=f'dropout_lstm_{i+1}')(x)

    # ── Fully-connected head ───────────────────────────────────────────────────
    for j, units in enumerate(dense_units):
        x = layers.Dense(units, activation='relu', name=f'dense_{j+1}')(x)
        x = layers.Dropout(dropout_rate, name=f'dropout_dense_{j+1}')(x)

    # ── Output layer ──────────────────────────────────────────────────────────
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    return keras.Model(inputs, outputs, name='LSTM_Gesture_Classifier')


model = build_lstm_model(
    input_shape       = (TARGET_LEN, NUM_FEATURES),
    num_classes       = NUM_CLASSES,
    lstm_units        = LSTM_UNITS,
    dropout_rate      = DROPOUT_RATE,
    recurrent_dropout = RECURRENT_DROPOUT,
    dense_units       = DENSE_UNITS,
)

model.summary()

---
## Cell 8 — Training

In [ ]:
"""
Compile and train the LSTM model.

Callbacks
---------
EarlyStopping     — restores best weights; stops after EARLY_STOP_PATIENCE
                    epochs of no improvement in val_loss.
ReduceLROnPlateau — halves the learning rate when val_loss plateaus for 5
                    epochs; minimum LR = 1e-6.
"""

# ── Compile ───────────────────────────────────────────────────────────────────
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy'],
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 5,
        min_lr   = 1e-6,
        verbose  = 1,
    ),
]

# ── Train ─────────────────────────────────────────────────────────────────────
print(f'Training on {len(X_train)} samples  |  '
      f'Validating on {len(X_val)} samples  |  '
      f'Max epochs: {EPOCHS}')

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

# ── Learning curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history.history['loss']) + 1)

# Loss
axes[0].plot(epochs_ran, history.history['loss'],     label='Train loss',
             color='#20808D', linewidth=2)
axes[0].plot(epochs_ran, history.history['val_loss'], label='Val loss',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history.history['accuracy'],     label='Train acc',
             color='#20808D', linewidth=2)
axes[1].plot(epochs_ran, history.history['val_accuracy'], label='Val acc',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History — LSTM Gesture Classifier', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'results/training_curves_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Training complete. Best val_loss: '
      f'{min(history.history["val_loss"]):.4f}  |  '
      f'Best val_acc: {max(history.history["val_accuracy"]):.4f}')

---
## Cell 9 — Evaluation

In [ ]:
"""
Evaluate the trained model on the held-out test set.

Outputs
-------
- Overall test accuracy and loss
- Per-class precision, recall, F1 (classification_report)
- Confusion matrix heatmap
"""

# ── Test loss and accuracy ─────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f'Test accuracy : {test_acc:.4f}  ({test_acc * 100:.2f}%)')
print(f'Test loss     : {test_loss:.4f}')

# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_proba = model.predict(X_test, verbose=0)          # (N, NUM_CLASSES)
y_pred_int   = np.argmax(y_pred_proba, axis=1)           # predicted class indices

# ── Classification report ─────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('Classification Report')
print('=' * 60)
report_str = classification_report(
    y_test_int, y_pred_int, target_names=CLASS_NAMES, zero_division=0
)
print(report_str)

# Store dict version for JSON saving later
report_dict = classification_report(
    y_test_int, y_pred_int, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test_int, y_pred_int)

# Normalise to percentages for readability
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES + 2), max(6, NUM_CLASSES)))

sns.heatmap(
    cm_norm,
    annot      = True,
    fmt        = '.1f',
    cmap       = 'Blues',
    xticklabels = CLASS_NAMES,
    yticklabels = CLASS_NAMES,
    linewidths  = 0.5,
    cbar_kws   = {'label': 'Recall (%)'},
    ax         = ax,
)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix — Test Accuracy {test_acc * 100:.2f}%',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'results/confusion_matrix_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Confusion matrix saved to results/confusion_matrix_{TIMESTAMP}.png')

---
## Cell 10 — Save Model and Results

In [ ]:
# =============================================================================
# CONFIGURATION  —  Edit this cell to control every aspect of the pipeline
# =============================================================================

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s = 31.2 Hz)
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz); set lower
                        # (e.g. 78) to halve resolution and speed up training.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations are valid — the feature list is built automatically.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (tip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment)

# IMU channel types (applies to finger IMUs AND palm_prox AND wrist)
USE_ACCELEROMETER = True  # ax, ay, az
USE_ORIENTATION   = True  # yaw, pitch, roll  (wrist uses 'heading' instead of 'yaw')

# Flex sensors  (MCP and PIP joints per finger)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU  (palm_prox — the real palm IMU; palm_mid is always zero → auto-excluded)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied to each channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove firmware;
# set 'none' to use them as-is, or apply an additional filter here.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalization applied after resampling
# Options: 'minmax' (→ [0,1]) | 'zscore' (zero-mean unit-var) | 'none'
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalize each trial independently (recommended)

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples (auto)
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
LSTM_UNITS        = [128, 64]  # Units per stacked LSTM layer
DROPOUT_RATE      = 0.3        # Dropout after each LSTM layer
RECURRENT_DROPOUT = 0.2        # Recurrent dropout within LSTM cells
DENSE_UNITS       = [64]       # Hidden FC units before softmax output

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 15


---
## Cell 11 — Inference Example *(optional)*

Load a single CSV trial, run it through the full preprocessing pipeline, and print the predicted gesture label.  
Set `INFERENCE_CSV` to the path of any valid glove CSV file.

In [ ]:
"""
Single-sample inference demo.

Steps
-----
1. Load the CSV with load_csv(), selecting only the feature columns used
   during training.
2. Apply the same temporal filter as training.
3. Resample to TARGET_LEN time steps.
4. Normalise with the same method as training.
5. Add a batch dimension and call model.predict().
6. Print the predicted class label and confidence scores.
"""

from utils.data_loader import (
    load_csv,
    preprocess_dataframe,
    resample_to_length,
    normalize,
)

# ── Set the path to the CSV you want to run inference on ──────────────────────
INFERENCE_CSV = '../data/example_gesture/trial_001.csv'  # <-- edit this path

# ── Check if file exists before proceeding ────────────────────────────────────
if not Path(INFERENCE_CSV).exists():
    print(f'[SKIP] Inference CSV not found: {INFERENCE_CSV}')
    print('Set INFERENCE_CSV to a valid path to run this cell.')
else:
    # 1. Load and select columns
    df_infer = load_csv(INFERENCE_CSV, feature_cols=feature_cols, exclude_quat=True)

    # Warn about any missing columns
    loaded_cols = df_infer.columns.tolist()
    missing_infer = [c for c in feature_cols if c not in loaded_cols]
    if missing_infer:
        print(f'WARNING: {len(missing_infer)} expected columns absent in inference CSV.')

    # 2. Temporal filter
    if FILTER_TYPE != 'none':
        df_infer = preprocess_dataframe(df_infer, filter_type=FILTER_TYPE, fs=FS_HZ)

    # 3. Resample
    arr_infer = df_infer.values.astype(np.float32)
    arr_infer = resample_to_length(arr_infer, TARGET_LEN)       # (TARGET_LEN, C)

    # 4. Normalise (per-sample: add and immediately remove the batch dim)
    arr_infer = arr_infer[np.newaxis, ...]                      # (1, T, C)
    if NORMALIZATION != 'none':
        arr_infer = normalize(arr_infer, method=NORMALIZATION, per_sample=True)

    # 5. Predict
    proba = model.predict(arr_infer, verbose=0)[0]              # (NUM_CLASSES,)
    pred_idx   = np.argmax(proba)
    pred_label = CLASS_NAMES[pred_idx]
    confidence = proba[pred_idx] * 100

    # 6. Report
    print(f'Input file   : {INFERENCE_CSV}')
    print(f'Predicted    : {pred_label}  (confidence {confidence:.1f}%)')
    print('\nPer-class probabilities:')
    for cls, p in sorted(zip(CLASS_NAMES, proba), key=lambda t: -t[1]):
        bar = '█' * int(p * 30)
        print(f'  {cls:<30} {p * 100:5.1f}%  {bar}')